In [29]:
import pandas as pd
import numpy as np
import os 

In [39]:
ds_years = [2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015]

In [40]:
# anoshift_path = "../datasets/Kyoto-2016_AnoShift/full/"
anoshift_path = "/Users/sakshamaggarwal/Downloads/Kyoto-2016_AnoShift/subset/"

In [41]:
df = pd.read_parquet(os.path.join(anoshift_path, f'{2006}_subset.parquet'))

In [42]:
print(df)

           0      1     2     3   4    5     6     7   8    9    10   11  \
0       c015  other   c20   c30   4  1.0   1.0   0.8  90  100   0.0  1.0   
1        c00  other   c20   c30   0  0.0   0.0   0.0   6   80  0.83  0.0   
2       c021   smtp  c274  c357   2  1.0   0.0   0.0   7   97   0.0  0.0   
3        c07   smtp  c274  c357  22  1.0   0.0   0.0  17   95   0.0  0.0   
4       c068   smtp  c220  c342   0  0.0   0.0   0.0   1   35   0.0  0.0   
...      ...    ...   ...   ...  ..  ...   ...   ...  ..  ...   ...  ...   
466769  c014  other   c20   c30   5  1.0   0.8   0.5   0  100   0.0  0.0   
466770   c04  other  c238  c338  34  1.0  0.32  0.33   0   58   0.0  0.0   
466771   c03  other  c245  c341   3  1.0   0.0   0.0  42   42   0.0  0.0   
466772  c014  other   c20   c30  77  1.0  0.88  0.71   0  100   0.0  0.0   
466773   c03  other   c20   c30   0  0.0   0.0   1.0  50   50   0.0  0.0   

          12    13                 14         15 16 17  18   19  
0        1.0    S0  2

In [43]:
def load_year(anoshift_db_path, year, valid=False):
    if valid:
        df = pd.read_parquet(os.path.join(anoshift_db_path, f'{year}_full_valid.parquet'))
    else:
        df = pd.read_parquet(os.path.join(anoshift_db_path, f'{year}_subset.parquet'))
    df = df.reset_index(drop=True)
    return df

In [44]:
def rename_columns(df):
    categorical_cols = ["0", "1", "2", "3", "13"]
    numerical_cols = ["4", "5", "6", "7", "8", "9", "10", "11", "12"]
    additional_cols = ["14", "15", "16", "17", "19"]
    label_col = ["18"]

    new_names = []
    for col_name in df.columns.values:
        if col_name in numerical_cols:
            df[col_name] = pd.to_numeric(df[col_name])
            new_names.append((col_name, "num_" + col_name))
        elif col_name in categorical_cols:
            new_names.append((col_name, "cat_" + col_name))
        elif col_name in additional_cols:
            new_names.append((col_name, "bonus_" + col_name))
        elif col_name in label_col:
            df[col_name] = pd.to_numeric(df[col_name])
            new_names.append((col_name, "label"))
        else:
            new_names.append((col_name, col_name))
    df.rename(columns=dict(new_names), inplace=True)
    return df

In [45]:
df_copy = rename_columns(df)
print(df_copy)

df_copy.loc[df['label'] < 0, 'label'] = -1
df_copy['label'].replace({1:0}, inplace=True)
df_copy['label'].replace({-1:1}, inplace=True)

print(df_copy)

       cat_0  cat_1 cat_2 cat_3  num_4  num_5  num_6  num_7  num_8  num_9  \
0       c015  other   c20   c30      4    1.0   1.00   0.80     90    100   
1        c00  other   c20   c30      0    0.0   0.00   0.00      6     80   
2       c021   smtp  c274  c357      2    1.0   0.00   0.00      7     97   
3        c07   smtp  c274  c357     22    1.0   0.00   0.00     17     95   
4       c068   smtp  c220  c342      0    0.0   0.00   0.00      1     35   
...      ...    ...   ...   ...    ...    ...    ...    ...    ...    ...   
466769  c014  other   c20   c30      5    1.0   0.80   0.50      0    100   
466770   c04  other  c238  c338     34    1.0   0.32   0.33      0     58   
466771   c03  other  c245  c341      3    1.0   0.00   0.00     42     42   
466772  c014  other   c20   c30     77    1.0   0.88   0.71      0    100   
466773   c03  other   c20   c30      0    0.0   0.00   1.00     50     50   

        num_10  num_11  num_12 cat_13           bonus_14   bonus_15 bonus_1

/var/folders/zc/6_ry_0zn42760dzx14r53c600000gn/T/ipykernel_49970/1563200969.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_copy['label'].replace({1:0}, inplace=True)
/var/folders/zc/6_ry_0zn42760dzx14r53c600000gn/T/ipykernel_49970/1563200969.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always 

In [46]:
label_counts = df_copy['label'].value_counts()
print("\n=== Label Distribution ===")
print(label_counts)


=== Label Distribution ===
label
1    416774
0     50000
Name: count, dtype: int64


In [47]:
# Initialize list to store results
year_stats = []

# Process each year
for year in range(2006, 2016):
    isValid = False
    # if year == 2010 or year == 2014 or year == 2015:
    #     isValid = True

    # Load and process data for each year    
    df = load_year(anoshift_path, year, isValid)
    df = rename_columns(df)
    
    # Transform labels
    df.loc[df['label'] < 0, 'label'] = -1
    df['label'].replace({1:0}, inplace=True)
    df['label'].replace({-1:1}, inplace=True)
    
    # Count labels
    label_counts = df['label'].value_counts()
    
    # Store results (handle cases where a label might not exist)
    count_0 = label_counts.get(0, 0)
    count_1 = label_counts.get(1, 0)
    total = count_0 + count_1
    
    year_stats.append({
        'Year': year,
        'Count_Label_0': count_0,
        'Count_Label_1': count_1,
        'Total': total,
    })

# Create DataFrame and save to CSV
results_df = pd.DataFrame(year_stats)
results_df.to_csv('label_distribution_by_year_(subset_files).csv', index=False)

# Display the results
print("\n=== Label Distribution By Year ===")
print(results_df)

/var/folders/zc/6_ry_0zn42760dzx14r53c600000gn/T/ipykernel_49970/3939597552.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['label'].replace({1:0}, inplace=True)
/var/folders/zc/6_ry_0zn42760dzx14r53c600000gn/T/ipykernel_49970/3939597552.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beh


=== Label Distribution By Year ===
   Year  Count_Label_0  Count_Label_1    Total
0  2006          50000         416774   466774
1  2007         300000         115471   415471
2  2008         300000          74713   374713
3  2009         299999         109403   409402
4  2010         299997         261262   561259
5  2011         299999         819248  1119247
6  2012         300000         634592   934592
7  2013         300000        1238064  1538064
8  2014         300000        1822662  2122662
9  2015         300000        1804322  2104322


/var/folders/zc/6_ry_0zn42760dzx14r53c600000gn/T/ipykernel_49970/3939597552.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['label'].replace({1:0}, inplace=True)
/var/folders/zc/6_ry_0zn42760dzx14r53c600000gn/T/ipykernel_49970/3939597552.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beh